# 1 · Extract a character's dialogue

Open in **Google Colab**, choose **Runtime → Change runtime type → GPU**. Run installation once, then **Runtime → Restart session** and continue at configuration, skipping installation. After restarting, run cells from configuration onward; running all cells reinstalls packages and requires another restart. The import cell checks the loaded and installed NumPy versions. Everything runs inside Colab; no paid inference API is used.

Accept the conditions for [pyannote Community-1](https://huggingface.co/pyannote/speaker-diarization-community-1), create a Hugging Face **read** token, and add it as the Colab secret `HF_TOKEN` with notebook access. This downloads model weights; the episode is processed in the Colab runtime.

First episode: choose an MP3 from Google Drive or upload one, and provide several **start/end** samples containing only your target voice, ideally 2–8 seconds each and varied delivery. Leave `PROFILE_FILE` empty to save/reuse `CHARACTER_ID.json` beside the episode, or find it one folder above. Later episodes: use the same `CHARACTER_ID` and leave `SAMPLE_TIMESTAMPS` empty. Download the ZIP before the runtime expires.

Diarization and speaker matching are estimates. Review the speaker table and audio previews before export. A profile describes a performed voice; a different character, whisper, or scream may need additional samples. Dialogue separation runs before diarization by default and reduces music/effects. All speakers remain in the dialogue stem, and separation can leave artifacts. Review the audio comparisons and target matches before export.

Keep generated archives, embeddings, and transcripts private. Before publishing a notebook, clear all outputs and remove personal inputs/tokens. Publishing code alone does not determine whether a particular use of source media is permitted.

In [ ]:
# Run ONLY in Colab. Restart afterward, then skip this cell.
import subprocess
import sys

if "google.colab" not in sys.modules:
    raise RuntimeError("Open this notebook in Google Colab before installing dependencies.")

required_packages: list[str] = [
    "numpy==2.2.6",
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
    "torchcodec==0.7.0",
    "pyannote.audio==4.0.4",
    "soundfile==0.13.1",
    "demucs==4.0.1",
    "dora-search==0.1.13",
    "openunmix==1.3.0",
    "lameenc==1.8.4",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *required_packages],
    check=True,
)

# Refresh NumPy's Python files and compiled extensions together, even if already installed.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
     "--force-reinstall", "--no-deps", "numpy==2.2.6"],
    check=True,
)
_VOICE_STAT_RESTART_REQUIRED: bool = True
print("Setup complete. Choose Runtime → Restart session, then continue at configuration.")

## Configure

IDs become metadata and directory names: use letters, digits, `_`, or `-`. Enter the credited actor yourself; the models cannot determine an actor's real-world identity. Times can be seconds, `MM:SS.sss`, or `HH:MM:SS.sss`.

`SPEAKER_THRESHOLD` is an uncalibrated cosine-similarity cutoff, **not a probability**. Inspect near matches and adjust it for your material. More than one cluster can match the same target. Override incorrect selections with `SPEAKER_OVERRIDE` after examining the results.

In [ ]:
from typing import TypeAlias

TimestampValue: TypeAlias = str | int | float

ACTOR_ID: str = "actor_x"
CHARACTER_ID: str = "character_x"
SERIES_ID: str = "series_x"
EPISODE_ID: str = "episode_01"
LANGUAGE: str = "ja"  # Spoken language, including the dub language when applicable.

# Set a Google Drive or existing Colab file path; empty opens the upload dialog.
# Example: "/content/drive/MyDrive/voice-stat/episodes/input_audio.mp3"
INPUT_AUDIO_FILE: str = ""
# Empty: look for CHARACTER_ID.json beside the audio, then one folder above.
# New profiles save beside the audio; discovered profiles update where found.
# An explicit path overrides lookup; that source file is left untouched.
# Example: "/content/drive/MyDrive/voice-stat/profiles/character_x.json"
PROFILE_FILE: str = ""
SAMPLE_TIMESTAMPS: list[tuple[TimestampValue, TimestampValue]] = [
    # ("00:01:15.679", "00:01:18.129"),
    # ("00:05:20.000", "00:05:24.000"),
]

SEPARATE_DIALOGUE: bool = True
# Save/reuse episode_diag.wav and episode_diag.json beside INPUT_AUDIO_FILE.
SEPARATION_SEGMENT_SECONDS: float = 6.0  # Reduce to 4.0 if GPU memory is insufficient.
SEPARATION_OVERLAP: float = 0.25  # More overlap takes longer; try 0.5 for difficult boundaries.

SPEAKER_THRESHOLD: float = 0.55
SPEAKER_OVERRIDE: list[str] | None = None  # None = automatic; e.g. ["SPEAKER_02", "SPEAKER_05"].
SILENCE_TOLERANCE_SECONDS: float = 2.0
BOUNDARY_PADDING_SECONDS: float = 0.08  # Exported as silence outside accepted speech.
VERIFICATION_WINDOW_SECONDS: float = 1.5
VERIFICATION_STEP_SECONDS: float = 0.5
MAX_SPEAKERS: int | None = None              # Set only if you have a useful upper bound.
EMBEDDING_SECONDS_PER_SPEAKER: float = 60.0
PREVIEW_CLIPS: int = 6
DIARIZATION_SEGMENTATION_BATCH_SIZE: int = 16
DIARIZATION_EMBEDDING_BATCH_SIZE: int = 16
EXPORT_WORKERS: int = 2  # Parallel FLAC encoding and hashing on the CPU; 1 disables concurrency.

DIARIZATION_MODEL: str = "pyannote/speaker-diarization-community-1"
DIARIZATION_REVISION: str = "3533c8cf8e369892e6b79ff1bf80f7b0286a54ee"
EMBEDDING_MODEL: str = "pyannote/wespeaker-voxceleb-resnet34-LM"
EMBEDDING_REVISION: str = "837717ddb9ff5507820346191109dc79c958d614"

print(f"[config] INPUT_AUDIO_FILE = {INPUT_AUDIO_FILE!r}")
print(f"[config] PROFILE_FILE = {PROFILE_FILE!r}")
print(f"[config] Voice sample timestamps: {len(SAMPLE_TIMESTAMPS)}")
print(f"[config] Dialogue separation: {SEPARATE_DIALOGUE}; reuse adjacent _diag.wav when present.")

In [ ]:
if globals().get("_VOICE_STAT_RESTART_REQUIRED", False):
    raise RuntimeError("Restart the Colab session after installation, then skip setup and continue at configuration.")

import gc
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import subprocess
import uuid
import zipfile
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable, TypeAlias, TypedDict
from time import perf_counter

os.environ["PYANNOTE_METRICS_ENABLED"] = "0"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import numpy as np

installed_numpy_version: str = importlib.metadata.version("numpy")
if np.__version__ != installed_numpy_version or installed_numpy_version != "2.2.6":
    raise RuntimeError(
        f"NumPy version mismatch: loaded {np.__version__}, installed {installed_numpy_version}; "
        "this notebook requires 2.2.6. Run setup, choose Runtime → Restart session, "
        "then continue at configuration, skipping setup."
    )

from numpy.typing import ArrayLike, NDArray
import pandas as pd
import soundfile as sf
import torch
from google.colab import drive, files, userdata
from IPython.display import Audio, display
from pyannote.audio import Inference, Model, Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook
from tqdm.auto import tqdm

JsonObject: TypeAlias = dict[str, Any]
FloatArray: TypeAlias = NDArray[np.float32]
Interval: TypeAlias = tuple[float, float]


class SpeakerTurn(TypedDict):
    start: float
    end: float
    speaker: str


class VerificationWindow(SpeakerTurn):
    embedding: FloatArray | None


class MergedSegment(TypedDict):
    start: float
    end: float
    speech_intervals: list[Interval]


if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime, then restart the session.")
if not shutil.which("ffmpeg"):
    raise RuntimeError("This runtime has no ffmpeg. Use a standard Colab runtime.")

for field, value in {
    "ACTOR_ID": ACTOR_ID, "CHARACTER_ID": CHARACTER_ID,
    "SERIES_ID": SERIES_ID, "EPISODE_ID": EPISODE_ID,
}.items():
    id_pattern: str = r"[A-Za-z0-9_-]+(?:\.json)?" if field == "CHARACTER_ID" else r"[A-Za-z0-9_-]+"
    if not re.fullmatch(id_pattern, value, flags=re.IGNORECASE):
        suffix_hint: str = " CHARACTER_ID may also end in .json." if field == "CHARACTER_ID" else ""
        raise ValueError(f"{field} must contain only letters, digits, underscores, or hyphens.{suffix_hint}")
if not -1 <= SPEAKER_THRESHOLD <= 1:
    raise ValueError("SPEAKER_THRESHOLD must be between -1 and 1.")
if min(SILENCE_TOLERANCE_SECONDS, BOUNDARY_PADDING_SECONDS) < 0:
    raise ValueError("Gap tolerance and padding must be nonnegative.")
if not 1 <= VERIFICATION_WINDOW_SECONDS <= 8:
    raise ValueError("VERIFICATION_WINDOW_SECONDS must be between 1 and 8 seconds.")
if not 0 < VERIFICATION_STEP_SECONDS <= VERIFICATION_WINDOW_SECONDS:
    raise ValueError("VERIFICATION_STEP_SECONDS must be positive and no larger than the window.")
if EMBEDDING_SECONDS_PER_SPEAKER < 1:
    raise ValueError("EMBEDDING_SECONDS_PER_SPEAKER must be at least 1.")
if min(DIARIZATION_SEGMENTATION_BATCH_SIZE, DIARIZATION_EMBEDDING_BATCH_SIZE, EXPORT_WORKERS) < 1:
    raise ValueError("Diarization batch sizes and export worker count must be positive.")

if SEPARATE_DIALOGUE:
    if not 1 <= SEPARATION_SEGMENT_SECONDS <= 7.8:
        raise ValueError("SEPARATION_SEGMENT_SECONDS must be between 1 and 7.8 seconds.")
    if not 0 <= SEPARATION_OVERLAP <= 0.8:
        raise ValueError("SEPARATION_OVERLAP must be between 0 and 0.8.")

DEVICE: torch.device = torch.device("cuda")
SAMPLE_RATE: int = 16000
print("GPU:", torch.cuda.get_device_name(0))


def write_json(path: Path, value: JsonObject) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_time(value: TimestampValue) -> float:
    if isinstance(value, (int, float)):
        seconds = float(value)
    else:
        parts = str(value).strip().split(":")
        if not 1 <= len(parts) <= 3:
            raise ValueError(f"Invalid timestamp: {value}")
        numbers = [float(part) for part in parts]
        if any(number < 0 for number in numbers):
            raise ValueError(f"Negative timestamp: {value}")
        if len(numbers) > 1 and any(number >= 60 for number in numbers[1:]):
            raise ValueError(f"Minutes/seconds outside timestamp range: {value}")
        seconds = sum(number * 60 ** index for index, number in enumerate(reversed(numbers)))
    if not math.isfinite(seconds) or seconds < 0:
        raise ValueError(f"Invalid timestamp: {value}")
    return seconds


def union_intervals(intervals: Iterable[Interval]) -> list[Interval]:
    merged: list[Interval] = []
    for start, end in sorted(intervals):
        if end <= start:
            continue
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(end, merged[-1][1]))
        else:
            merged.append((float(start), float(end)))
    return merged


def subtract_intervals(start: float, end: float, blockers: list[Interval]) -> list[Interval]:
    pieces: list[Interval] = []
    cursor = start
    for left, right in blockers:
        if right <= cursor:
            continue
        if left >= end:
            break
        if left > cursor:
            pieces.append((cursor, min(left, end)))
        cursor = max(cursor, right)
        if cursor >= end:
            break
    if cursor < end:
        pieces.append((cursor, end))
    return pieces


def overlap_seconds(start: float, end: float, intervals: list[Interval]) -> float:
    return sum(max(0.0, min(end, right) - max(start, left)) for left, right in intervals)

## Load and decode

Set `INPUT_AUDIO_FILE` to a full Google Drive path such as `/content/drive/MyDrive/voice-stat/episodes/input_audio.mp3`. Drive mounts automatically when the audio or profile path uses `/content/drive`; approve Colab's Drive access prompt. The episode is copied to temporary Colab storage for processing, so you can reuse the Drive file across sessions.

Leave `INPUT_AUDIO_FILE` empty to upload exactly one audio file. With `PROFILE_FILE` empty, lookup checks `CHARACTER_ID.json` beside the audio, then exactly one folder above; an existing `.json` suffix is kept. If neither exists, an uploaded profile JSON can be used, or samples create a new profile beside the audio. New samples update an automatically discovered profile where it was found. An explicit `PROFILE_FILE` overrides lookup and leaves that source file untouched. The standalone profile download is also named `CHARACTER_ID.json`, and the ZIP includes it as `profile/CHARACTER_ID.json`.

In [ ]:
print(f"[input] Extraction notebook: selecting episode audio (kernel PID {os.getpid()}).")
print(f"[input] INPUT_AUDIO_FILE = {INPUT_AUDIO_FILE!r}")
print(f"[input] PROFILE_FILE = {PROFILE_FILE!r}")
print(f"[input] Working directory: {Path.cwd()}")
drive_root: Path = Path("/content/drive")
uses_drive: bool = any(
    Path(path).is_relative_to(drive_root)
    for path in (INPUT_AUDIO_FILE, PROFILE_FILE)
    if path
)
drive_mounted: bool = drive_root.is_mount()
print(f"[input] Configured paths use Google Drive: {uses_drive}; Drive mounted: {drive_mounted}")
if uses_drive and not drive_mounted:
    print("[input] Mounting Google Drive; Colab may show a Google account authorization prompt.", flush=True)
    drive.mount(str(drive_root))
    print(f"[input] Drive mounted after authorization: {drive_root.is_mount()}")

uploaded_names: list[str] = []
if not INPUT_AUDIO_FILE:
    print(
        "[input] Opening browser upload because INPUT_AUDIO_FILE is empty. "
        "Set the INPUT_AUDIO_FILE assignment in the configuration cell and run that cell. "
        "Rerunning configuration resets the variable to the value written there; "
        "PROFILE_FILE specifies only the speaker profile.",
        flush=True,
    )
    uploaded = files.upload()
    uploaded_names: list[str] = list(uploaded)
    print(f"[input] Uploaded filenames: {uploaded_names!r}")
    audio_names: list[str] = [name for name in uploaded_names if Path(name).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".ogg"}]
    if len(audio_names) != 1:
        raise ValueError("Upload exactly one episode audio file.")
    INPUT_AUDIO_FILE: str = audio_names[0]
    del uploaded  # Release the extra in-memory upload copy.
else:
    print(f"[input] Using configured audio path: {INPUT_AUDIO_FILE!r}; skipping browser upload.")

audio_path: Path = Path(INPUT_AUDIO_FILE).absolute()
print(f"[input] Checking episode audio file: {audio_path.absolute()}", flush=True)
if not audio_path.is_file():
    raise FileNotFoundError(
        f"Episode audio file not found: {audio_path}. Parent directory exists: {audio_path.parent.is_dir()}. "
        "Use a full file path such as /content/drive/MyDrive/folder/episode.mp3, not a Drive sharing URL."
    )
print(f"[input] Audio file found: {audio_path.stat().st_size / (1024 * 1024):.2f} MiB")

profile_filename: str = CHARACTER_ID if Path(CHARACTER_ID).suffix.lower() == ".json" else f"{CHARACTER_ID}.json"
print(f"[profile] Profile output filename: {profile_filename}")
profile_input_path: Path | None = Path(PROFILE_FILE).absolute() if PROFILE_FILE else None
profile_output_path: Path | None = None
if PROFILE_FILE:
    print(f"[profile] Using explicit PROFILE_FILE: {profile_input_path}; automatic lookup skipped.")
else:
    profile_output_path = audio_path.parent / profile_filename
    profile_candidates: list[Path] = [profile_output_path, audio_path.parent.parent / profile_filename]
    for candidate in profile_candidates:
        print(f"[profile] Checking automatic profile path: {candidate}", flush=True)
        if candidate.is_file():
            profile_input_path = candidate
            profile_output_path = candidate
            print(f"[profile] Found {profile_filename}: {candidate}")
            break
    if profile_input_path is None:
        profile_names: list[str] = [name for name in uploaded_names if Path(name).suffix.lower() == ".json"]
        if len(profile_names) > 1:
            raise ValueError("Choose one speaker profile with PROFILE_FILE.")
        if profile_names:
            profile_input_path = Path(profile_names[0]).absolute()
            print(f"[profile] Using uploaded profile: {profile_input_path}")
        else:
            print("[profile] No existing profile found; sample timestamps are required to create one.")
    print(f"[profile] Automatic profile save path: {profile_output_path}")
print(f"[input] Selected audio: {audio_path}; profile: {profile_input_path}")
print(f"[input] Voice sample timestamps: {len(SAMPLE_TIMESTAMPS)}")
if not SAMPLE_TIMESTAMPS and profile_input_path is None:
    raise ValueError(
        f"No speaker profile was found. Add SAMPLE_TIMESTAMPS to create {profile_output_path}, "
        "or set PROFILE_FILE to an existing profile."
    )
if profile_input_path is not None:
    print(f"[input] Checking speaker profile file: {profile_input_path}", flush=True)
    if not profile_input_path.is_file():
        raise FileNotFoundError(f"Speaker profile file not found: {profile_input_path}. Use its full Colab or Google Drive path.")
    print("[input] Speaker profile file found.")

run_id: str = uuid.uuid4().hex[:12]
run_root: Path = Path("/content/voice-stat") / SERIES_ID / EPISODE_ID / CHARACTER_ID / run_id
run_root.mkdir(parents=True, exist_ok=False)
bundle_root: Path = run_root / "extraction"
bundle_root.mkdir()
(bundle_root / "segments").mkdir()
(bundle_root / "profile").mkdir()
(bundle_root / "review").mkdir()

local_audio_path: Path = audio_path
if audio_path.is_relative_to(drive_root):
    local_audio_path = run_root / f"source_audio{audio_path.suffix}"
    print(f"[input] Copying Drive audio to {local_audio_path}...", flush=True)
    copy_started: float = perf_counter()
    shutil.copyfile(audio_path, local_audio_path)
    print(f"[input] Drive copy completed in {perf_counter() - copy_started:.2f}s.")
print(f"[input] Hashing audio at {local_audio_path}...", flush=True)
hash_started: float = perf_counter()
source_sha256: str = file_sha256(local_audio_path)
print(f"[input] Audio hash completed in {perf_counter() - hash_started:.2f}s: {source_sha256[:12]}")

decoded_path: Path = run_root / "episode_16k_mono.wav"
print(f"[input] Decoding audio to {SAMPLE_RATE} Hz mono: {decoded_path}...", flush=True)
decode_started: float = perf_counter()
command: list[str] = ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
           "-i", str(local_audio_path), "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE),
           "-c:a", "pcm_s16le", str(decoded_path)]
result: subprocess.CompletedProcess[str] = subprocess.run(command, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(f"Audio conversion failed: {result.stderr[-2000:]}")
print(f"[input] Audio conversion completed in {perf_counter() - decode_started:.2f}s.")
audio: FloatArray
rate: int
audio, rate = sf.read(decoded_path, dtype="float32")
duration = len(audio) / rate
if rate != SAMPLE_RATE or audio.ndim != 1 or duration <= 0:
    raise ValueError("Expected a nonempty 16 kHz mono decoded episode.")

original_audio: FloatArray = audio  # Retain the mixture for comparison and separation retries.
print("[input] Sample previews below use the original mixture; the next cell separates dialogue.")

samples: list[Interval] = []
for start_value, end_value in SAMPLE_TIMESTAMPS:
    start, end = parse_time(start_value), parse_time(end_value)
    if not 0 <= start < end <= duration:
        raise ValueError(f"Sample {start_value}–{end_value} is outside the {duration:.3f}s episode.")
    if end - start < 1:
        raise ValueError("Choose voice samples at least one second long; 2–8 seconds is preferable.")
    samples.append((start, end))
    display(Audio(audio[round(start * rate):round(end * rate)], rate=rate))

print(f"Episode duration: {duration / 60:.2f} minutes; source hash: {source_sha256[:12]}")

## Separate dialogue from music and effects

Enabled by default. [MVSEP CDX23](https://github.com/ZFTurbo/MVSEP-CDX23-Cinematic-Sound-Demixing) is a Demucs model trained for three sources: dialogue, effects, and music. This cell keeps dialogue from all speakers; the following cells identify your target. It uses the original stereo audio for separation, then creates a 16 kHz mono dialogue track on the original timeline. No silence is removed.

Listen to the original/dialogue comparisons. Separation can leave background sounds or alter speech, including tone and emotion cues. Set `SEPARATE_DIALOGUE = False` to process the original mixture. Existing profiles remain usable, but review matches when their samples used different preprocessing.

For `INPUT_AUDIO_FILE = "/content/drive/MyDrive/folder/episode.mp3"`, the entire cleaned episode is saved in the same folder as `episode_diag.wav`, with separation metadata in `episode_diag.json`. These filenames use the episode name regardless of `CHARACTER_ID`. Only the reusable speaker profile uses `CHARACTER_ID.json`. Existing dialogue is reused without GPU separation when its metadata identifies the current episode; a cache from another episode is replaced. Files without provenance can still be reused when their audio format and timeline match. To regenerate dialogue, rename or remove the `_diag.wav` file. WAV avoids another lossy MP3 encode. Local Colab inputs use a local sidecar that expires with the runtime. Only target clips go into the extraction ZIP. New dependencies install in Colab's setup cell; install once and restart before continuing.

In [ ]:
audio = original_audio
preprocessing_settings: JsonObject = {"mode": "original_mixture"}
separation_metadata: JsonObject | None = None


def separate_episode_dialogue(output_path: Path) -> JsonObject:
    from demucs.apply import apply_model
    from demucs.htdemucs import HTDemucs
    from demucs.states import load_model

    checkpoint_name: str = "97d170e1-dbb4db15.th"
    expected_checkpoint_sha256: str = "dbb4db154df7e45a5cb72d1659c48937e757f6d6b0eef8ca4199e6e38f8d8f37"
    checkpoint_url: str = (
        "https://github.com/ZFTurbo/MVSEP-CDX23-Cinematic-Sound-Demixing/"
        f"releases/download/v.1.0.0/{checkpoint_name}"
    )
    model_root: Path = Path("/content/voice-stat-models")
    model_root.mkdir(parents=True, exist_ok=True)
    checkpoint_path: Path = model_root / checkpoint_name
    if not checkpoint_path.is_file():
        print(f"[separation] Downloading dialogue model: {checkpoint_name}", flush=True)
        torch.hub.download_url_to_file(
            checkpoint_url, str(checkpoint_path), hash_prefix=expected_checkpoint_sha256, progress=True,
        )
    checkpoint_sha256: str = file_sha256(checkpoint_path)
    if checkpoint_sha256 != expected_checkpoint_sha256:
        raise RuntimeError(f"Dialogue checkpoint checksum mismatch: {checkpoint_path}. Remove it and retry.")

    # This pinned Demucs checkpoint contains its model class and constructor metadata.
    checkpoint_package: dict[str, Any] = torch.load(
        checkpoint_path, map_location="cpu", weights_only=False,
    )
    separation_model: HTDemucs = load_model(checkpoint_package)
    del checkpoint_package
    maximum_segment: float = float(separation_model.segment)
    if SEPARATION_SEGMENT_SECONDS > maximum_segment:
        raise ValueError(f"This model supports separation segments up to {maximum_segment}s.")
    separation_model.segment = SEPARATION_SEGMENT_SECONDS
    separation_model.eval()
    separation_rate: int = separation_model.samplerate
    separation_channels: int = separation_model.audio_channels
    temporary_root: Path = run_root / "dialogue-separation"
    temporary_root.mkdir(exist_ok=True)
    stereo_path: Path = temporary_root / "input_stereo.wav"
    dialogue_path: Path = temporary_root / "dialogue_native.wav"
    resampled_path: Path = temporary_root / "dialogue_16k_mono.wav"

    print(f"[separation] Decoding original audio at {separation_rate} Hz, {separation_channels} channels.", flush=True)
    decode_result: subprocess.CompletedProcess[str] = subprocess.run(
        ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
         "-i", str(local_audio_path), "-vn", "-ac", str(separation_channels),
         "-ar", str(separation_rate), "-c:a", "pcm_f32le", str(stereo_path)],
        capture_output=True, text=True,
    )
    if decode_result.returncode:
        raise RuntimeError(f"Separation input conversion failed: {decode_result.stderr[-2000:]}")

    stereo_audio: FloatArray
    stereo_rate: int
    stereo_audio, stereo_rate = sf.read(stereo_path, dtype="float32", always_2d=True)
    # Keep the full episode and returned stems on CPU; only inference windows use GPU memory.
    mixture: torch.Tensor = torch.from_numpy(stereo_audio.T).unsqueeze(0)
    print(
        f"[separation] Separating {len(stereo_audio) / stereo_rate / 60:.2f} minutes; "
        f"segment={SEPARATION_SEGMENT_SECONDS}s, overlap={SEPARATION_OVERLAP}, shifts=0.",
        flush=True,
    )
    started: float = perf_counter()
    try:
        with torch.inference_mode():
            separated: torch.Tensor = apply_model(
                separation_model, mixture, device=DEVICE, split=True, shifts=0,
                segment=SEPARATION_SEGMENT_SECONDS, overlap=SEPARATION_OVERLAP,
                num_workers=0, progress=True,
            )
        # The author's checkpoint/inference uses source order: music, effects, dialogue.
        dialogue_mono: torch.Tensor = separated[0, 2].mean(dim=0)
        sf.write(dialogue_path, dialogue_mono.numpy(), separation_rate, subtype="FLOAT")
        del separated, dialogue_mono, mixture, stereo_audio
    finally:
        separation_model.cpu()
        del separation_model
        gc.collect()
        torch.cuda.empty_cache()
    print(f"[separation] GPU separation completed in {perf_counter() - started:.1f}s.", flush=True)

    # Resampling can differ by a rounding sample; preserve the exact original 16 kHz frame count.
    timeline_filter: str = (
        f"aresample={rate},apad=whole_len={len(original_audio)},"
        f"atrim=end_sample={len(original_audio)}"
    )
    resample_result: subprocess.CompletedProcess[str] = subprocess.run(
        ["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
         "-i", str(dialogue_path), "-af", timeline_filter, "-ac", "1", "-ar", str(rate),
         "-c:a", "pcm_f32le", str(resampled_path)],
        capture_output=True, text=True,
    )
    if resample_result.returncode:
        raise RuntimeError(f"Dialogue resampling failed: {resample_result.stderr[-2000:]}")
    dialogue_audio: FloatArray
    dialogue_rate: int
    dialogue_audio, dialogue_rate = sf.read(resampled_path, dtype="float32")
    peak: float = float(np.max(np.abs(dialogue_audio)))
    gain: float = 0.99 / peak if peak > 0.99 else 1.0
    if gain < 1.0:
        dialogue_audio *= gain  # Attenuate globally only when needed to prevent PCM clipping.
        sf.write(resampled_path, dialogue_audio, dialogue_rate, subtype="FLOAT")

    temporary_cache_path: Path = output_path.with_name(f"{output_path.stem}.{run_id}.tmp.wav")
    print(f"[separation] Saving cleaned episode to {output_path}...", flush=True)
    shutil.copyfile(resampled_path, temporary_cache_path)
    temporary_cache_path.replace(output_path)
    metadata: JsonObject = {
        "checkpoint": checkpoint_name, "checkpoint_url": checkpoint_url,
        "checkpoint_sha256": checkpoint_sha256, "stem": "dialogue", "stem_index": 2,
        "native_sample_rate": separation_rate, "output_sample_rate": dialogue_rate,
        "output_frames": len(dialogue_audio), "peak_before_gain": peak, "global_gain": gain,
        "audio_sha256": file_sha256(output_path),
    }
    stereo_path.unlink()
    dialogue_path.unlink()
    resampled_path.unlink()
    return metadata


if SEPARATE_DIALOGUE:
    requested_preprocessing: JsonObject = {
        "mode": "dialogue_separation", "model": "MVSEP-CDX23/97d170e1-dbb4db15",
        "demucs_version": importlib.metadata.version("demucs"), "sample_rate": rate,
        "segment_seconds": SEPARATION_SEGMENT_SECONDS, "overlap": SEPARATION_OVERLAP,
        "shifts": 0, "peak_limit": 0.99,
    }
    dialogue_cache_path: Path = audio_path.with_name(f"{audio_path.stem}_diag.wav")
    dialogue_metadata_path: Path = dialogue_cache_path.with_suffix(".json")
    print(f"[separation] Checking dialogue cache: {dialogue_cache_path}", flush=True)
    reuse_dialogue_cache: bool = dialogue_cache_path.is_file()
    cached_metadata: JsonObject | None = None
    if reuse_dialogue_cache:
        if dialogue_metadata_path.is_file():
            try:
                cached_metadata = json.loads(dialogue_metadata_path.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError) as error:
                print(f"[separation] Provenance could not be read: {error}")
        if cached_metadata is not None and cached_metadata.get("source_sha256") not in (None, source_sha256):
            print("[separation] Cache belongs to another episode; regenerating dialogue.")
            reuse_dialogue_cache = False
    if reuse_dialogue_cache:
        cache_info: Any = sf.info(dialogue_cache_path)
        if cache_info.samplerate != rate or cache_info.channels != 1 or cache_info.frames != len(original_audio):
            print("[separation] Cache format/timeline differs from this episode; regenerating dialogue.")
            reuse_dialogue_cache = False
    if reuse_dialogue_cache:
        existing_dialogue_sha256: str = file_sha256(dialogue_cache_path)
        if cached_metadata is not None and cached_metadata.get("audio_sha256") != existing_dialogue_sha256:
            print("[separation] Provenance checksum differs; using the WAV with unknown model provenance.")
            cached_metadata = None
        print(f"[separation] Reusing {dialogue_cache_path}; skipping GPU separation.")
        preprocessing_settings = {
            "mode": "reused_dialogue_stem", "audio_sha256": existing_dialogue_sha256,
        }
        if cached_metadata is not None:
            separation_metadata = cached_metadata
            preprocessing_settings = cached_metadata.get("preprocessing", preprocessing_settings)
            if preprocessing_settings != requested_preprocessing:
                print("[separation] Reusing the WAV's saved separation settings. Rename/remove it to regenerate with current settings.")
        else:
            print("[separation] Using the existing WAV with unknown model provenance.")
    else:
        preprocessing_settings = requested_preprocessing
        separation_metadata = separate_episode_dialogue(dialogue_cache_path)
        separation_metadata["source_sha256"] = source_sha256
        separation_metadata["preprocessing"] = preprocessing_settings
        temporary_metadata_path: Path = dialogue_metadata_path.with_name(f"{dialogue_metadata_path.stem}.{run_id}.tmp.json")
        write_json(temporary_metadata_path, separation_metadata)
        temporary_metadata_path.replace(dialogue_metadata_path)
        print(f"[separation] Dialogue episode saved beside input: {dialogue_cache_path}")
        print(f"[separation] Provenance saved: {dialogue_metadata_path}")
    dialogue_audio, dialogue_rate = sf.read(dialogue_cache_path, dtype="float32")
    if dialogue_rate != rate or dialogue_audio.ndim != 1 or len(dialogue_audio) != len(original_audio):
        raise RuntimeError(
            f"Dialogue file {dialogue_cache_path} must match the original mono timeline: "
            f"expected {rate} Hz and {len(original_audio)} frames; "
            f"got {dialogue_rate} Hz, {len(dialogue_audio)} frames, {dialogue_audio.ndim} dimensions."
        )
    audio = dialogue_audio
    preview_intervals: list[Interval] = samples[:PREVIEW_CLIPS] or [(0.0, min(duration, 12.0))]
    for start, end in preview_intervals:
        print(f"[separation] Original mixture: {start:.3f}s–{end:.3f}s")
        display(Audio(original_audio[round(start * rate):round(end * rate)], rate=rate))
        print("[separation] Separated dialogue:")
        display(Audio(audio[round(start * rate):round(end * rate)], rate=rate))
    print(f"[separation] Full dialogue WAV: {dialogue_cache_path}")
    print("[separation] Profiling, diarization, and exported clips will use separated dialogue.")
else:
    print("[separation] Disabled; profiling, diarization, and clips will use the original mixture.")

## Diarize the whole episode

This estimates speaker turns, including overlap. A cluster can contain mistakes: profile samples choose the target voice but do not correct diarization boundaries. Set `MAX_SPEAKERS` only when you know a useful upper bound.

The two `DIARIZATION_*_BATCH_SIZE` settings control GPU batches. Reduce them after a memory error. Matching also computes short overlapping voice embeddings for the verification step, reusing the loaded model and embedding batch size. Changing the verification window/step requires rerunning matching; changing selection or score cutoffs only requires rerunning export.

In [ ]:
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError("Add the HF_TOKEN read token to Colab Secrets and enable notebook access.") from error

pipeline = Pipeline.from_pretrained(
    DIARIZATION_MODEL, revision=DIARIZATION_REVISION, token=hf_token,
)
if pipeline is None:
    raise RuntimeError("Model access failed. Accept Community-1 conditions and check your HF token.")
pipeline.to(DEVICE)
pipeline.segmentation_batch_size = DIARIZATION_SEGMENTATION_BATCH_SIZE
pipeline.embedding_batch_size = DIARIZATION_EMBEDDING_BATCH_SIZE
kwargs: dict[str, int] = {} if MAX_SPEAKERS is None else {"max_speakers": MAX_SPEAKERS}
diarization_started: float = perf_counter()
with ProgressHook() as hook:
    diarization_output = pipeline(
        {"waveform": torch.from_numpy(audio).unsqueeze(0), "sample_rate": rate},
        hook=hook, **kwargs,
    )

turns: list[SpeakerTurn] = sorted([
    {"start": max(0.0, float(turn.start)), "end": min(duration, float(turn.end)), "speaker": speaker}
    for turn, speaker in diarization_output.speaker_diarization
    if min(duration, turn.end) > max(0.0, turn.start)
], key=lambda turn: (turn["start"], turn["end"], turn["speaker"]))
if not turns:
    raise RuntimeError("No speech was detected in the episode.")
write_json(bundle_root / "diarization.json", {"schema_version": 1, "turns": turns})

del pipeline, diarization_output
gc.collect()
torch.cuda.empty_cache()
print(f"Detected {len(turns)} turns across {len({turn['speaker'] for turn in turns})} clusters "
      f"in {perf_counter() - diarization_started:.1f}s (excluding model download/load).")

In [ ]:
profile: JsonObject | None = None
if profile_input_path is not None:
    profile: JsonObject | None = json.loads(profile_input_path.read_text(encoding="utf-8"))
    if profile.get("schema_version") != 1 or profile.get("kind") != "voice_stat_speaker_profile":
        raise ValueError("This is not a compatible voice-stat speaker profile.")
    if profile["actor_id"] != ACTOR_ID or profile["character_id"] != CHARACTER_ID:
        raise ValueError("Set ACTOR_ID and CHARACTER_ID to the saved profile's IDs.")
    if profile["embedding_model"] != EMBEDDING_MODEL:
        raise ValueError("Saved profile uses a different embedding model.")
    EMBEDDING_REVISION: str = profile["embedding_revision"]  # Keep the embedding space identical.
    if any(
        source.get("preprocessing", {"mode": "original_mixture"}) != preprocessing_settings
        for source in profile["sample_sources"]
    ):
        print("[profile] Saved samples used different audio preprocessing. Review matches or add new samples.")

embedding_model = Model.from_pretrained(
    EMBEDDING_MODEL, revision=EMBEDDING_REVISION, token=userdata.get("HF_TOKEN"),
)
if embedding_model is None:
    raise RuntimeError("Speaker embedding model could not be loaded.")
embedding_inference = Inference(embedding_model, window="whole")
embedding_inference.to(DEVICE)


def normalize_embedding(vector: ArrayLike) -> FloatArray:
    embedding: FloatArray = np.asarray(vector, dtype=np.float32).reshape(-1)
    norm = float(np.linalg.norm(embedding))
    if not np.isfinite(embedding).all() or norm == 0:
        raise ValueError("Invalid embedding; choose a longer, clearer voice sample.")
    return embedding / norm


def embed_interval(start: float, end: float) -> FloatArray:
    clip: FloatArray = audio[round(start * rate):round(end * rate)]
    with torch.inference_mode():
        vector = embedding_inference({"waveform": torch.from_numpy(clip).unsqueeze(0), "sample_rate": rate})
    return normalize_embedding(vector)


exemplars: list[FloatArray] = [] if profile is None else [normalize_embedding(vector) for vector in profile["exemplars"]]
for start, end in tqdm(samples, desc="Profile samples"):
    # Bound inference size without discarding any of the supplied sample.
    count = max(1, math.ceil((end - start) / 8))
    edges = np.linspace(start, end, count + 1)
    for left, right in zip(edges[:-1], edges[1:]):
        exemplars.append(embed_interval(float(left), float(right)))
reference_vectors: FloatArray = np.stack(exemplars)
if profile and reference_vectors.shape[1] != profile["embedding_dimension"]:
    raise ValueError("Saved profile embedding dimensions do not match.")
reference_centroid: FloatArray = normalize_embedding(reference_vectors.mean(axis=0))
if profile is None:
    profile: JsonObject | None = {
        "schema_version": 1, "kind": "voice_stat_speaker_profile",
        "profile_id": uuid.uuid4().hex,
        "actor_id": ACTOR_ID, "character_id": CHARACTER_ID,
        "embedding_model": EMBEDDING_MODEL, "embedding_revision": EMBEDDING_REVISION,
        "created_utc": datetime.now(timezone.utc).isoformat(), "sample_sources": [],
    }
profile.update({
    "embedding_dimension": int(reference_vectors.shape[1]),
    "exemplars": reference_vectors.tolist(), "centroid": reference_centroid.tolist(),
})
if samples:
    profile["sample_sources"].append({
        "source_sha256": source_sha256, "intervals_seconds": samples,
        "preprocessing": preprocessing_settings,
    })
profile_bundle_path: Path = bundle_root / "profile" / profile_filename
write_json(profile_bundle_path, profile)
if profile_output_path is not None and (samples or profile_input_path != profile_output_path):
    print(f"[profile] Saving standalone profile to {profile_output_path}...", flush=True)
    temporary_profile_path: Path = profile_output_path.with_name(f"{profile_output_path.stem}.{run_id}.tmp.json")
    write_json(temporary_profile_path, profile)
    temporary_profile_path.replace(profile_output_path)
    print(f"[profile] Standalone profile saved: {profile_output_path}")


def make_verification_windows(start: float, end: float, speaker: str) -> list[VerificationWindow]:
    first_sample, last_sample = round(start * rate), round(end * rate)
    window_samples = min(round(VERIFICATION_WINDOW_SECONDS * rate), last_sample - first_sample)
    step_samples = round(VERIFICATION_STEP_SECONDS * rate)
    count = max(1, math.ceil((last_sample - first_sample - window_samples) / step_samples) + 1)
    starts = np.linspace(first_sample, last_sample - window_samples, count).round().astype(int)
    return [
        {"start": int(left) / rate, "end": int(left + window_samples) / rate,
         "speaker": speaker, "embedding": None}
        for left in starts
    ]


cluster_rows: list[JsonObject] = []
cluster_centroids: dict[str, FloatArray] = {}
verification_windows: list[VerificationWindow] = []
speaker_examples: dict[str, Interval] = {}
matching_started: float = perf_counter()
for speaker in tqdm(sorted({turn["speaker"] for turn in turns}), desc="Match speaker clusters"):
    speaker_turns: list[SpeakerTurn] = [turn for turn in turns if turn["speaker"] == speaker]
    other_intervals: list[Interval] = union_intervals([(turn["start"], turn["end"]) for turn in turns if turn["speaker"] != speaker])
    clean_pieces: list[Interval] = []
    for turn in speaker_turns:
        clean_pieces.extend(subtract_intervals(turn["start"], turn["end"], other_intervals))
    clean_pieces: list[Interval] = union_intervals(clean_pieces)
    for left, right in clean_pieces:
        verification_windows.extend(make_verification_windows(left, right, speaker))
    windows: list[Interval] = []
    for left, right in clean_pieces:
        count = max(1, math.ceil((right - left) / 8))
        edges = np.linspace(left, right, count + 1)
        windows.extend((float(a), float(b)) for a, b in zip(edges[:-1], edges[1:]) if b - a >= 1)
    windows.sort(key=lambda window: window[1] - window[0], reverse=True)
    vectors: list[FloatArray] = []
    weights: list[float] = []
    used_seconds = 0.0
    for left, right in windows:
        remaining = EMBEDDING_SECONDS_PER_SPEAKER - used_seconds
        if remaining < 1:
            break
        right = min(right, left + remaining)
        vectors.append(embed_interval(left, right))
        weights.append(right - left)
        used_seconds += right - left
    centroid_similarity: float | None = None
    exemplar_similarity: float | None = None
    score: float | None = None
    longest_turn = max(speaker_turns, key=lambda turn: turn["end"] - turn["start"])
    speaker_examples[speaker] = (longest_turn["start"], min(longest_turn["end"], longest_turn["start"] + 8))
    if vectors:
        cluster_centroid = normalize_embedding(np.average(np.stack(vectors), axis=0, weights=weights))
        cluster_centroids[speaker] = cluster_centroid
        centroid_similarity = float(cluster_centroid @ reference_centroid)
        exemplar_similarity = float(np.max(reference_vectors @ cluster_centroid))
        score = max(centroid_similarity, exemplar_similarity)
        speaker_examples[speaker] = windows[0]
    # Samples identify known clusters only when one cluster occupies >=80% and others <=10%.
    seed_hits = 0
    for start, end in samples:
        own_intervals = union_intervals([(turn["start"], turn["end"]) for turn in speaker_turns])
        own_fraction = overlap_seconds(start, end, own_intervals) / (end - start)
        other_fraction = overlap_seconds(start, end, other_intervals) / (end - start)
        if own_fraction >= 0.8 and other_fraction <= 0.1:
            seed_hits += 1
    cluster_rows.append({
        "speaker": speaker, "similarity": score, "centroid_similarity": centroid_similarity,
        "exemplar_similarity": exemplar_similarity, "sample_hits": seed_hits,
        "embedded_seconds": used_seconds,
        "speech_seconds": sum(right - left for left, right in union_intervals([(turn["start"], turn["end"]) for turn in speaker_turns])),
        "automatic_match": bool(seed_hits or (score is not None and score >= SPEAKER_THRESHOLD)),
    })

cluster_table: pd.DataFrame = pd.DataFrame(cluster_rows).sort_values("similarity", ascending=False, na_position="last")
cluster_table.to_csv(bundle_root / "review" / "speaker_matches.csv", index=False)
display(cluster_table)
# Batch equal-length windows without adding silence or a neighbor's voice to short turns.
# Sub-second pieces stay unverified; a cluster label alone cannot establish their identity.
windows_by_length: dict[int, list[VerificationWindow]] = {}
for window in verification_windows:
    frame_count = round(window["end"] * rate) - round(window["start"] * rate)
    if frame_count >= rate:
        windows_by_length.setdefault(frame_count, []).append(window)

with tqdm(total=sum(map(len, windows_by_length.values())), desc="Verify short voice windows") as progress:
    for same_length in windows_by_length.values():
        for offset in range(0, len(same_length), DIARIZATION_EMBEDDING_BATCH_SIZE):
            batch: list[VerificationWindow] = same_length[offset:offset + DIARIZATION_EMBEDDING_BATCH_SIZE]
            waveforms: FloatArray = np.stack([
                audio[round(window["start"] * rate):round(window["end"] * rate)]
                for window in batch
            ])
            batch_embeddings: FloatArray = embedding_inference.infer(torch.from_numpy(waveforms).unsqueeze(1))
            for window, vector in zip(batch, batch_embeddings):
                # Undefined model outputs are reviewable rejections, never target matches.
                if np.isfinite(vector).all() and np.linalg.norm(vector) > 0:
                    window["embedding"] = normalize_embedding(vector)
            progress.update(len(batch))

del embedding_inference, embedding_model
gc.collect()
torch.cuda.empty_cache()
print(f"Speaker-cluster matching: {perf_counter() - matching_started:.1f}s.")

In [ ]:
# Preview any cluster without rerunning GPU matching. Empty = the closest matches.
PREVIEW_SPEAKERS: list[str] = []  # e.g. ["SPEAKER_02", "SPEAKER_05"]
preview_speakers = PREVIEW_SPEAKERS or cluster_table.head(PREVIEW_CLIPS)["speaker"].tolist()
for speaker in preview_speakers:
    if speaker not in speaker_examples:
        raise ValueError(f"Unknown speaker ID: {speaker}")
    left, right = speaker_examples[speaker]
    print(speaker, "similarity:", cluster_table.loc[cluster_table["speaker"] == speaker, "similarity"].iloc[0])
    display(Audio(audio[round(left * rate):round(right * rate)], rate=rate))

## Select, verify, merge, and export

Set `SPEAKER_OVERRIDE` after listening. Each candidate is checked against the saved target samples and unselected speaker centroids using short overlapping windows. A window must meet `VERIFY_THRESHOLD` and beat the closest other speaker by `VERIFY_MARGIN`. These are uncalibrated cutoffs, not probabilities. This additional filter can still miss another voice; it is not speaker separation.

Detected other-speaker speech, rejected windows, and `EXCLUDE_TIMESTAMPS` are removed before merging and always force a split. A rejected window wins over an overlapping accepted window. Unverifiable pieces shorter than one second are excluded; review the rejected audio before drawing statistical conclusions, since this favors cleaner clips over completeness.

The two-second tolerance applies only to quiet gaps (20 ms RMS below `GAP_MAX_DBFS`). Louder unlabeled gaps split clips. Retained gaps and boundary padding are written as silence with their original duration, so they cannot bring unverified voices back into the export. Speech inside accepted intervals is unchanged.

Rerun this cell and ZIP download after adjusting selection, cutoffs, or episode-time exclusions; no GPU matching rerun is needed. Review controls below show clip duration, clip-relative timestamps, and rejection reasons. Counts represent clips, not grammatical sentences.

In [ ]:
# Set this here after reviewing the speaker table, if necessary.
# SPEAKER_OVERRIDE = ["SPEAKER_02"]
records: list[JsonObject] = []  # A failed rerun must not download the previous selection.
VERIFY_THRESHOLD: float = 0.55
VERIFY_MARGIN: float = 0.05
GAP_MAX_DBFS: float = -45.0  # Louder gaps are not assumed to be silence.
EXCLUDE_TIMESTAMPS: list[tuple[TimestampValue, TimestampValue]] = [
    # Episode times, not offsets within an exported clip. Review below prints conversions.
    # ("00:15.300", "00:17.000"),
]

if not -1 <= VERIFY_THRESHOLD <= 1 or not 0 <= VERIFY_MARGIN <= 2:
    raise ValueError("VERIFY_THRESHOLD must be in [-1, 1] and VERIFY_MARGIN in [0, 2].")
if not math.isfinite(GAP_MAX_DBFS) or GAP_MAX_DBFS > 0:
    raise ValueError("GAP_MAX_DBFS must be a finite value at or below 0 dBFS.")
excluded_intervals: list[Interval] = []
for start_value, end_value in EXCLUDE_TIMESTAMPS:
    start, end = parse_time(start_value), parse_time(end_value)
    if not 0 <= start < end <= duration:
        raise ValueError(f"Excluded interval {start_value}–{end_value} is outside this episode.")
    excluded_intervals.append((start, end))

known_speakers: set[str] = {row["speaker"] for row in cluster_rows}
selected_speakers: set[str] = (
    {row["speaker"] for row in cluster_rows if row["automatic_match"]}
    if SPEAKER_OVERRIDE is None else set(SPEAKER_OVERRIDE)
)
if not selected_speakers <= known_speakers:
    raise ValueError(f"Unknown speaker IDs: {selected_speakers - known_speakers}")
if not selected_speakers:
    raise ValueError("No target matched. Review the table and adjust the threshold or selection.")
print(f"[export] Selected speaker clusters: {', '.join(sorted(selected_speakers))}")

other_intervals: list[Interval] = union_intervals([(turn["start"], turn["end"]) for turn in turns if turn["speaker"] not in selected_speakers])
rival_speakers: list[str] = sorted(set(cluster_centroids) - selected_speakers)
rival_vectors: FloatArray | None = (
    np.stack([cluster_centroids[speaker] for speaker in rival_speakers]) if rival_speakers else None
)
verification_rows: list[JsonObject] = []
accepted_intervals: list[Interval] = []
rejected_intervals: list[Interval] = []
for window in verification_windows:
    if window["speaker"] not in selected_speakers:
        continue
    vector = window["embedding"]
    target_score: float | None = None
    rival_score: float | None = None
    rival_speaker: str | None = None
    reason: str = "too_short" if window["end"] - window["start"] < 1 else "invalid_embedding"
    if vector is not None:
        target_score = max(float(vector @ reference_centroid), float(np.max(reference_vectors @ vector)))
        if rival_vectors is not None:
            rival_scores = rival_vectors @ vector
            best_rival = int(np.argmax(rival_scores))
            rival_score = float(rival_scores[best_rival])
            rival_speaker = rival_speakers[best_rival]
        if target_score < VERIFY_THRESHOLD:
            reason = "weak_target_match"
        elif rival_score is not None and target_score - rival_score < VERIFY_MARGIN:
            reason = "ambiguous_or_other_speaker"
        else:
            reason = "accepted"
    if reason == "accepted":
        accepted_intervals.append((window["start"], window["end"]))
    else:
        rejected_intervals.append((window["start"], window["end"]))
    verification_rows.append({
        "start": window["start"], "end": window["end"], "speaker": window["speaker"],
        "target_similarity": target_score, "rival_similarity": rival_score,
        "rival_speaker": rival_speaker, "decision": reason,
    })

blocked_intervals: list[Interval] = union_intervals([
    *other_intervals, *rejected_intervals, *excluded_intervals,
])
target_intervals: list[Interval] = [
    piece for left, right in union_intervals(accepted_intervals)
    for piece in subtract_intervals(left, right, blocked_intervals)
]

verification_report: JsonObject = {
    "schema_version": 1, "source_sha256": source_sha256,
    "window_seconds": VERIFICATION_WINDOW_SECONDS, "step_seconds": VERIFICATION_STEP_SECONDS,
    "threshold": VERIFY_THRESHOLD, "margin": VERIFY_MARGIN,
    "selected_speakers": sorted(selected_speakers), "windows": verification_rows,
    "excluded_intervals": excluded_intervals,
}
write_json(bundle_root / "review" / "window_verification.json", verification_report)
print("[verification] Window decisions (overlapping window counts, not dialogue counts):")
display(pd.Series([row["decision"] for row in verification_rows]).value_counts().rename("windows"))


def gap_is_quiet(start: float, end: float) -> bool:
    gap_audio = audio[round(start * rate):round(end * rate)]
    frame_size = round(0.02 * rate)
    maximum_rms = 10 ** (GAP_MAX_DBFS / 20)
    for offset in range(0, len(gap_audio), frame_size):
        frame = gap_audio[offset:offset + frame_size]
        if float(np.sqrt(np.mean(frame ** 2))) > maximum_rms:
            return False
    return True


merged_segments: list[MergedSegment] = []
for left, right in target_intervals:
    if merged_segments:
        previous = merged_segments[-1]
        gap = left - previous["end"]
        blocked_gap = overlap_seconds(previous["end"], left, blocked_intervals) > 0
        if gap <= SILENCE_TOLERANCE_SECONDS and not blocked_gap and gap_is_quiet(previous["end"], left):
            previous["end"] = max(previous["end"], right)
            previous["speech_intervals"].append((left, right))
            continue
    merged_segments.append({"start": left, "end": right, "speech_intervals": [(left, right)]})
kept_seconds: float = sum(right - left for left, right in target_intervals)
print(f"[export] {len(target_intervals)} intervals after verification and exclusions -> "
      f"{len(merged_segments)} clips, {kept_seconds:.3f}s of accepted speech.")


def export_segment(task: tuple[int, MergedSegment]) -> JsonObject | None:
    index, segment = task
    left, right = segment["start"], segment["end"]
    clip_start = max(0.0, left - BOUNDARY_PADDING_SECONDS)
    clip_end = min(duration, right + BOUNDARY_PADDING_SECONDS)
    # Do not pad into another clip or newly include non-target speech.
    if index > 1:
        clip_start = max(clip_start, (merged_segments[index - 2]["end"] + left) / 2)
    if index < len(merged_segments):
        clip_end = min(clip_end, (right + merged_segments[index]["start"]) / 2)
    for other_start, other_end in blocked_intervals:
        if other_start < left and other_end > clip_start:
            clip_start = max(clip_start, min(left, other_end))
        if other_end > right and other_start < clip_end:
            clip_end = min(clip_end, max(right, other_start))
    start_sample, end_sample = round(clip_start * rate), round(clip_end * rate)
    if end_sample <= start_sample:
        return None
    clip_start, clip_end = start_sample / rate, end_sample / rate
    identity = json.dumps([source_sha256, ACTOR_ID, CHARACTER_ID, start_sample, end_sample])
    observation_id = hashlib.sha256(identity.encode()).hexdigest()[:24]
    filename = f"segment_{index:04d}_{start_sample:010d}-{end_sample:010d}.flac"
    relative_path = f"segments/{filename}"
    # Keep episode timing, but copy only accepted speech. Gaps/padding stay silent.
    clip_audio: FloatArray = np.zeros(end_sample - start_sample, dtype=np.float32)
    for speech_start, speech_end in segment["speech_intervals"]:
        first = max(start_sample, round(speech_start * rate))
        last = min(end_sample, round(speech_end * rate))
        clip_audio[first - start_sample:last - start_sample] = audio[first:last]
    sf.write(bundle_root / relative_path, clip_audio, rate, subtype="PCM_16")
    flags: list[str] = []
    overlap = overlap_seconds(clip_start, clip_end, other_intervals)
    speech_seconds = sum(b - a for a, b in union_intervals(segment["speech_intervals"]))
    if overlap > 0:
        flags.append("overlapping_non_target_speech")
    if speech_seconds < 1:
        flags.append("short_speech")
    if right - left > 30:
        flags.append("long_clip_may_contain_multiple_emotions")
    return {
        "observation_id": observation_id, "actor_id": ACTOR_ID, "character_id": CHARACTER_ID,
        "series_id": SERIES_ID, "episode_id": EPISODE_ID, "language": LANGUAGE,
        "source_sha256": source_sha256, "audio_file": relative_path,
        "audio_sha256": file_sha256(bundle_root / relative_path),
        "start_seconds": clip_start, "end_seconds": clip_end,
        "speech_start_seconds": left, "speech_end_seconds": right,
        "speech_intervals": segment["speech_intervals"], "speech_seconds": speech_seconds,
        "clip_seconds": clip_end - clip_start, "overlap_seconds": overlap,
        "speaker_clusters": sorted({
            turn["speaker"] for turn in turns
            if turn["speaker"] in selected_speakers and turn["start"] < right and turn["end"] > left
        }),
        "quality_flags": flags,
    }


export_started: float = perf_counter()
with ThreadPoolExecutor(max_workers=EXPORT_WORKERS) as executor:
    exported_records = executor.map(export_segment, enumerate(merged_segments, start=1))
    records = [
        record for record in tqdm(exported_records, total=len(merged_segments), desc="Export FLAC clips")
        if record is not None
    ]
if not records:
    raise ValueError(
        "No target audio passed verification. Review rejected windows below, the profile samples, "
        "and selected speakers before adjusting VERIFY_THRESHOLD or VERIFY_MARGIN."
    )
print(f"Audio export: {perf_counter() - export_started:.1f}s with {EXPORT_WORKERS} workers.")

# On export, only current records are included; earlier previews are not added to the ZIP.
write_json(bundle_root / "segments.json", {"schema_version": 1, "segments": records})
manifest: JsonObject = {
    "schema_version": 1, "kind": "voice_stat_extraction", "run_id": run_id,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "actor_id": ACTOR_ID, "character_id": CHARACTER_ID, "series_id": SERIES_ID,
    "episode_id": EPISODE_ID, "language": LANGUAGE,
    "source_sha256": source_sha256, "source_duration_seconds": duration,
    "sample_rate": rate, "channels": 1, "audio_format": "FLAC PCM_16",
    "models": {"diarization": {"id": DIARIZATION_MODEL, "revision": DIARIZATION_REVISION},
               "embedding": {"id": EMBEDDING_MODEL, "revision": EMBEDDING_REVISION}},
    "settings": {"speaker_threshold": SPEAKER_THRESHOLD, "selected_speakers": sorted(selected_speakers),
                 "selection_method": "manual_override" if SPEAKER_OVERRIDE is not None else "similarity_and_sample_hits",
                 "silence_tolerance_seconds": SILENCE_TOLERANCE_SECONDS,
                 "padding_seconds": BOUNDARY_PADDING_SECONDS,
                 "include_overlapping_speech": False,
                 "verification_window_seconds": VERIFICATION_WINDOW_SECONDS,
                 "verification_step_seconds": VERIFICATION_STEP_SECONDS,
                 "verification_threshold": VERIFY_THRESHOLD, "verification_margin": VERIFY_MARGIN,
                 "gap_max_dbfs": GAP_MAX_DBFS, "gaps_and_padding": "silence",
                 "excluded_intervals": excluded_intervals,
                 "embedding_seconds_per_speaker": EMBEDDING_SECONDS_PER_SPEAKER,
                 "diarization_segmentation_batch_size": DIARIZATION_SEGMENTATION_BATCH_SIZE,
                 "diarization_embedding_batch_size": DIARIZATION_EMBEDDING_BATCH_SIZE,
                 "export_workers": EXPORT_WORKERS,
                 "audio_preprocessing": preprocessing_settings},
    "package_versions": {name: importlib.metadata.version(name) for name in ["numpy", "torch", "torchaudio", "pyannote.audio", "soundfile", "demucs", "dora-search", "openunmix"]},
    "segment_count": len(records), "segments_file": "segments.json",
    "profile_file": profile_bundle_path.relative_to(bundle_root).as_posix(),
}
if separation_metadata is not None:
    manifest["models"]["dialogue_separation"] = {
        key: value for key, value in separation_metadata.items()
        if key not in {"source_sha256", "preprocessing"}
    }
write_json(bundle_root / "manifest.json", manifest)
display(pd.DataFrame(records)[["observation_id", "start_seconds", "end_seconds", "speech_seconds", "quality_flags"]])
for clip_number, record in enumerate(records[:PREVIEW_CLIPS], start=1):
    print(f"Clip {clip_number}: {record['audio_file']} | {record['clip_seconds']:.3f}s | "
          f"episode {record['start_seconds']:.3f}–{record['end_seconds']:.3f}s | {record['quality_flags']}")
    display(Audio(filename=str(bundle_root / record["audio_file"])))

In [ ]:
# Optional review: rerun freely without GPU work or changing exported files.
# For rejected windows, set REVIEW_REJECTED = True; CLIP_NUMBER then selects a rejected range.
REVIEW_REJECTED: bool = False
CLIP_NUMBER: int = 1
REMOVE_FROM_CLIP: list[tuple[float, float]] = []  # e.g. [(5.2, 6.8)]; seconds in this player.


def review_extraction(number: int, rejected: bool, removals: list[tuple[float, float]]) -> None:
    if rejected:
        intervals = union_intervals(rejected_intervals)
        if not 1 <= number <= len(intervals):
            print(f"There are {len(intervals)} rejected ranges. Choose CLIP_NUMBER within that range.")
            return
        start, end = intervals[number - 1]
        print(f"Rejected range {number}/{len(intervals)} | {end - start:.3f}s | episode {start:.3f}–{end:.3f}s")
        display(Audio(audio[round(start * rate):round(end * rate)], rate=rate))
    else:
        if not 1 <= number <= len(records):
            print(f"There are {len(records)} exported clips. Choose CLIP_NUMBER within that range.")
            return
        record = records[number - 1]
        start, end = record["start_seconds"], record["end_seconds"]
        print(f"File: {record['audio_file']}")
        print(f"Duration: {record['clip_seconds']:.3f}s | episode {start:.3f}–{end:.3f}s")
        display(Audio(filename=str(bundle_root / record["audio_file"])))

    for title, rows in [("Diarization labels", turns), ("Window verification", verification_rows)]:
        print(title, "(episode times and offsets within this player):")
        display(pd.DataFrame([
            {**row, "clip_start": max(start, row["start"]) - start,
             "clip_end": min(end, row["end"]) - start}
            for row in rows if row["start"] < end and row["end"] > start
        ]))
    for left, right in removals:
        if not 0 <= left < right <= end - start:
            raise ValueError("REMOVE_FROM_CLIP must lie within the displayed player duration.")
        print(f"Add to EXCLUDE_TIMESTAMPS in export: ({start + left:.6f}, {start + right:.6f}),")
    if removals:
        print("Rerun export and ZIP download after copying these episode times. Clip numbers may change.")


review_extraction(CLIP_NUMBER, REVIEW_REJECTED, REMOVE_FROM_CLIP)

In [ ]:
if not records:
    raise RuntimeError("Run export successfully before downloading the ZIP.")
archive_path: Path = run_root / f"{SERIES_ID}_{EPISODE_ID}_{CHARACTER_ID}_extraction.zip"
included_files: list[Path] = [bundle_root / record["audio_file"] for record in records]
included_files += [bundle_root / name for name in [
    "manifest.json", "segments.json", "diarization.json",
    "review/speaker_matches.csv", "review/window_verification.json",
]]
included_files.append(profile_bundle_path)
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in included_files:
        archive.write(path, arcname=str(Path("extraction") / path.relative_to(bundle_root)))
print(f"Exported {len(records)} clips; ZIP size: {archive_path.stat().st_size / 1024**2:.1f} MiB")
files.download(str(archive_path))

# Save this standalone file for the next episode (also included inside the ZIP).
files.download(str(profile_bundle_path))